In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [17]:
def _get_project_home() -> Path:
    """
    Determine project root directory. Works whether script is run from
    trace-analysis/ or project root.
    """
    # First, check if current working directory is the project root
    cwd = Path.cwd()
    if (cwd / "traces").exists():
        return cwd
    
    # Otherwise, derive from script location
    # Script is at trace-analysis/analyze.py, so project root is parent
    script_dir = Path.cwd()  # In notebook, use cwd
    project_root = script_dir.parent if script_dir.name == "trace-analysis" else script_dir
    
    # Verify traces/ exists
    if (project_root / "traces").exists():
        return project_root
    
    # Fallback: return parent anyway (will fail later with clear error)
    return project_root

PROJECT_HOME = _get_project_home()

def get_csv_path(dataset_number: int) -> Path:
    return (
        PROJECT_HOME
        / "traces"
        / "alibaba"
        / "cluster-trace-microservices-v2022"
        / "data"
        / "CallGraph"
        / f"CallGraph_{dataset_number}.csv"
    )

def _read_one(path: str | Path, **read_csv_kwargs) -> pd.DataFrame:
    return pd.read_csv(path, **read_csv_kwargs)

def _parent_rpc_id(rpc_id: str | float | int | None) -> str | None:
    """
    Return the parent RPC id for a dotted rpc_id string.
    Example: '0.1.2' -> '0.1'. Roots (no dot) return None.
    """
    if rpc_id is None or (isinstance(rpc_id, float) and np.isnan(rpc_id)):
        return None
    rpc_str = str(rpc_id).strip()
    if not rpc_str or "." not in rpc_str:
        return None
    return rpc_str.rsplit(".", 1)[0]

def sample_traces(df: pd.DataFrame, fraction: float, trace_col: str = "traceid", random_state: int | None = None) -> pd.DataFrame:
    """
    Sample a fraction of traces from the dataframe.
    
    For rows belonging to the same trace, either keep them all or drop them all
    (maintains trace integrity).
    
    Args:
        df: Input dataframe with trace data
        fraction: Fraction of traces to sample (0.0 to 1.0)
        trace_col: Column name containing trace IDs (default: "traceid")
        random_state: Random seed for reproducibility (default: None)
    
    Returns:
        DataFrame containing all rows for the sampled traces
    """
    if fraction <= 0.0 or fraction > 1.0:
        raise ValueError(f"fraction must be in (0.0, 1.0], got {fraction}")
    
    if trace_col not in df.columns:
        raise ValueError(f"Column '{trace_col}' not found in dataframe")
    
    # Get unique trace IDs
    unique_traces = df[trace_col].dropna().unique()
    
    if len(unique_traces) == 0:
        print("[WARN] No valid trace IDs found")
        return pd.DataFrame()
    
    # Sample trace IDs
    n_samples = max(1, int(len(unique_traces) * fraction))
    sampled_trace_ids = pd.Series(unique_traces).sample(
        n=n_samples, 
        random_state=random_state
    ).values
    
    # Filter dataframe to keep all rows for sampled traces
    sampled_df = df[df[trace_col].isin(sampled_trace_ids)].copy()
    
    print(f"Sampled {len(sampled_trace_ids):,} traces ({fraction*100:.1f}%) from {len(unique_traces):,} unique traces")
    print(f"Result: {len(sampled_df):,} rows from {len(df):,} original rows")
    
    return sampled_df

def print_sibling_window_overlap_stats(df: pd.DataFrame) -> None:
    """
    Determine whether sibling RPCs (children of the same parent within a trace)
    execute sequentially by checking if their [start, end) windows overlap.
    """
    required_cols = {"traceid", "rpc_id", "timestamp", "rt"}
    missing = required_cols - set(df.columns)
    if missing:
        print(f"[WARN] Cannot compute sibling window stats, missing columns: {sorted(missing)}")
        return

    base = (
        df.loc[:, ["traceid", "rpc_id", "timestamp", "rt"]]
        .dropna(subset=["traceid", "rpc_id", "timestamp", "rt"])
        .copy()
    )
    if base.empty:
        print("[WARN] No rows with complete (traceid, rpc_id, timestamp, rt) data for sibling analysis.")
        return

    base["rpc_id"] = base["rpc_id"].astype(str).str.strip()
    base = base[base["rpc_id"] != ""]
    if base.empty:
        print("[WARN] No valid rpc_id values remain after cleaning for sibling analysis.")
        return

    base["timestamp"] = base["timestamp"].astype(float)
    base["rt"] = base["rt"].astype(float)
    base["end_timestamp"] = base["timestamp"] + base["rt"]
    base["parent_rpc_id"] = base["rpc_id"].map(_parent_rpc_id)

    siblings = base[base["parent_rpc_id"].notna()]
    if siblings.empty:
        print("[INFO] No sibling relationships detected (no parent rpc ids).")
        return

    grouped = siblings.groupby(["traceid", "parent_rpc_id"])

    total_groups = 0
    non_overlap_groups = 0
    overlap_groups = 0
    total_children = 0
    non_overlap_children = 0
    overlap_children = 0
    tol = 1e-9

    for (_, _), group in grouped:
        if len(group) < 2:
            continue
        total_groups += 1
        total_children += len(group)

        ordered = group.sort_values("timestamp")
        prev_end = None
        sequential = True
        for _, row in ordered.iterrows():
            start = float(row["timestamp"])
            end = float(row["end_timestamp"])
            if prev_end is None:
                prev_end = end
                continue
            if start < prev_end - tol:
                sequential = False
                break
            prev_end = max(prev_end, end)

        if sequential:
            non_overlap_groups += 1
            non_overlap_children += len(group)
        else:
            overlap_groups += 1
            overlap_children += len(group)

    if total_groups == 0:
        print("[INFO] No parent nodes have two or more children to compare.")
        return

    pct_groups_seq = (non_overlap_groups / total_groups) * 100.0
    pct_groups_overlap = 100.0 - pct_groups_seq
    pct_children_seq = (non_overlap_children / total_children) * 100.0 if total_children else 0.0
    pct_children_overlap = 100.0 - pct_children_seq

    print("Sibling window overlap:")
    print(f"  Parent nodes with >=2 children: {total_groups:,}")
    print(f"  Sequential (non-overlapping) parents: {non_overlap_groups:,} ({pct_groups_seq:.2f}%)")
    print(f"  Overlapping parents: {overlap_groups:,} ({pct_groups_overlap:.2f}%)")
    print(f"  Child calls under analyzed parents: {total_children:,}")
    print(f"  Children in sequential groups: {non_overlap_children:,} ({pct_children_seq:.2f}%)")
    print(f"  Children in overlapping groups: {overlap_children:,} ({pct_children_overlap:.2f}%)")

def _build_call_pattern_for_group(group: pd.DataFrame, tol: float = 1e-9) -> list:
    """
    Build a call pattern from a group of sibling RPCs.
    
    Returns a sequential list where each element is either:
    - A dict with single RPC info (sequential call)
    - A list of dicts (fanout/parallel calls)
    """
    # Sort by timestamp
    ordered = group.sort_values("timestamp").copy()
    
    call_pattern = []
    current_fanout = []
    prev_end = None
    
    for _, row in ordered.iterrows():
        start = float(row["timestamp"])
        end = float(row["end_timestamp"])
        rpc_info = {
            "rpc_id": str(row["rpc_id"]),
            "dm": str(row["dm"]),
            "interface": str(row.get("interface", "")) if pd.notna(row.get("interface")) else "",
            "timestamp": start,
            "rt": float(row["rt"]),
            "end_timestamp": end,
        }
        
        if prev_end is None:
            # First call - start a new group
            current_fanout = [rpc_info]
            prev_end = end
        elif start < prev_end - tol:
            # Overlaps with previous - add to current fanout
            current_fanout.append(rpc_info)
            prev_end = max(prev_end, end)
        else:
            # Sequential - finish current group and start new one
            if len(current_fanout) == 1:
                call_pattern.append(current_fanout[0])  # Single sequential call
            else:
                call_pattern.append(current_fanout)  # Fanout group
            current_fanout = [rpc_info]
            prev_end = end
    
    # Add the last group
    if current_fanout:
        if len(current_fanout) == 1:
            call_pattern.append(current_fanout[0])
        else:
            call_pattern.append(current_fanout)
    
    return call_pattern

def _normalize_pattern(pattern: list) -> tuple:
    """
    Normalize a call pattern to a canonical representation for comparison.
    Returns a tuple representation that can be used as a dictionary key.
    """
    normalized = []
    for item in pattern:
        if isinstance(item, list):
            # Fanout: sort by (dm, interface) for consistency
            fanout_items = sorted(
                [(rpc["dm"], rpc.get("interface", "")) for rpc in item],
                key=lambda x: (x[0], x[1])
            )
            normalized.append(("fanout", tuple(fanout_items)))
        else:
            # Sequential call
            normalized.append(("seq", (item["dm"], item.get("interface", ""))))
    return tuple(normalized)

def analyze_parent_child_call_patterns(df: pd.DataFrame) -> dict[str, dict[str, dict]]:
    """
    Analyze call patterns for each service, aggregating across all traces.
    
    For each service and each parent RPC, determine how it calls its children,
    aggregated across all traces that belong to the same service.
    
    Args:
        df: DataFrame with RPC data (must have columns: service, traceid, rpc_id, timestamp, rt)
    
    Returns:
        Dictionary structure: {
            service_name: {
                parent_rpc_id: {
                    "patterns": [list of all unique patterns],
                    "pattern_freq": {normalized_pattern: count},
                    "most_common": most_common_pattern,
                    "total_occurrences": total_count
                }
            }
        }
        where each pattern is a list where each element is either:
        - A dict with single RPC info (sequential call)
        - A list of dicts (fanout/parallel calls)
    """
    required_cols = {"service", "traceid", "rpc_id", "timestamp", "rt"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")
    
    # Prepare base data
    base = (
        df.loc[:, ["service", "traceid", "rpc_id", "timestamp", "rt", "dm", "interface"]]
        .dropna(subset=["traceid", "rpc_id", "timestamp", "rt"])
        .copy()
    )
    
    if base.empty:
        return {}
    
    base["rpc_id"] = base["rpc_id"].astype(str).str.strip()
    base = base[base["rpc_id"] != ""]
    if base.empty:
        return {}
    
    base["timestamp"] = base["timestamp"].astype(float)
    base["rt"] = base["rt"].astype(float)
    base["end_timestamp"] = base["timestamp"] + base["rt"]
    base["parent_rpc_id"] = base["rpc_id"].map(_parent_rpc_id)
    
    # Filter to only rows with parent (siblings)
    siblings = base[base["parent_rpc_id"].notna()].copy()
    if siblings.empty:
        return {}
    
    # First, compute patterns per (service, traceid, parent_rpc_id)
    per_trace_patterns: dict[str, dict[str, list]] = {}
    tol = 1e-9
    
    for (service, traceid, parent_rpc_id), group in siblings.groupby(["service", "traceid", "parent_rpc_id"]):
        if service not in per_trace_patterns:
            per_trace_patterns[service] = {}
        
        pattern = _build_call_pattern_for_group(group, tol)
        key = f"{traceid}:{parent_rpc_id}"
        per_trace_patterns[service][key] = pattern
    
    # Now aggregate by (service, parent_rpc_id) across all traces
    result: dict[str, dict[str, dict]] = {}
    
    for service, trace_patterns in per_trace_patterns.items():
        if service not in result:
            result[service] = {}
        
        # Group by parent_rpc_id (extract from key)
        parent_patterns: dict[str, list] = {}
        for key, pattern in trace_patterns.items():
            # key is "traceid:parent_rpc_id", extract parent_rpc_id
            parent_rpc_id = key.split(":", 1)[1]
            if parent_rpc_id not in parent_patterns:
                parent_patterns[parent_rpc_id] = []
            parent_patterns[parent_rpc_id].append(pattern)
        
        # For each parent_rpc_id, aggregate patterns
        for parent_rpc_id, patterns_list in parent_patterns.items():
            # Count pattern frequencies
            pattern_freq: dict[tuple, int] = {}
            pattern_examples: dict[tuple, list] = {}
            
            for pattern in patterns_list:
                normalized = _normalize_pattern(pattern)
                pattern_freq[normalized] = pattern_freq.get(normalized, 0) + 1
                if normalized not in pattern_examples:
                    pattern_examples[normalized] = pattern
            
            # Find most common pattern
            if pattern_freq:
                most_common_norm = max(pattern_freq.items(), key=lambda x: x[1])[0]
                most_common_pattern = pattern_examples[most_common_norm]
            else:
                most_common_pattern = []
            
            result[service][parent_rpc_id] = {
                "patterns": list(pattern_examples.values()),  # All unique patterns
                "pattern_freq": {str(k): v for k, v in pattern_freq.items()},  # Frequency counts
                "most_common": most_common_pattern,
                "total_occurrences": len(patterns_list)
            }
    
    return result

def print_call_pattern_summary(patterns: dict[str, dict[str, dict]]) -> None:
    """
    Print summary statistics about call patterns (aggregated per service).
    """
    total_parents = 0
    total_sequential_only = 0
    total_with_fanouts = 0
    total_calls = 0
    total_sequential_calls = 0
    total_fanout_calls = 0
    fanout_sizes = []
    total_pattern_variations = 0
    
    for service, service_patterns in patterns.items():
        for parent_rpc_id, parent_data in service_patterns.items():
            total_parents += 1
            most_common = parent_data["most_common"]
            total_pattern_variations += len(parent_data["patterns"])
            
            has_fanout = any(isinstance(item, list) for item in most_common)
            
            if has_fanout:
                total_with_fanouts += 1
            else:
                total_sequential_only += 1
            
            for item in most_common:
                if isinstance(item, list):
                    # Fanout
                    fanout_size = len(item)
                    fanout_sizes.append(fanout_size)
                    total_fanout_calls += fanout_size
                else:
                    # Sequential call
                    total_sequential_calls += 1
            
            total_calls += len(most_common)
    
    print("Call Pattern Summary (aggregated per service):")
    print(f"  Total services analyzed: {len(patterns):,}")
    print(f"  Total unique parent RPCs: {total_parents:,}")
    print(f"  Parents with only sequential calls: {total_sequential_only:,}")
    print(f"  Parents with at least one fanout: {total_with_fanouts:,}")
    print(f"  Total unique pattern variations: {total_pattern_variations:,}")
    print(f"  Total call groups in most common patterns: {total_calls:,}")
    print(f"  Sequential call groups: {total_sequential_calls:,}")
    print(f"  Fanout groups: {len(fanout_sizes):,}")
    print(f"  Total RPCs in fanouts: {total_fanout_calls:,}")
    
    if fanout_sizes:
        print(f"  Fanout size statistics:")
        print(f"    Min: {min(fanout_sizes)}")
        print(f"    Max: {max(fanout_sizes)}")
        print(f"    Mean: {np.mean(fanout_sizes):.2f}")
        print(f"    Median: {np.median(fanout_sizes):.2f}")

def get_simplified_call_patterns(patterns: dict[str, dict[str, dict]]) -> dict[str, dict[str, list]]:
    """
    Get a simplified view of call patterns - just the most common pattern per parent.
    
    Returns:
        Dictionary structure: {
            service_name: {
                parent_rpc_id: [most_common_call_pattern]
            }
        }
    """
    simplified = {}
    for service, service_patterns in patterns.items():
        simplified[service] = {
            parent_rpc_id: parent_data["most_common"]
            for parent_rpc_id, parent_data in service_patterns.items()
        }
    return simplified

In [3]:
# Read one CSV file (dataset 0)
csv_path = get_csv_path(0)
print(f"Reading CSV file: {csv_path}")

df_orig = _read_one(csv_path, on_bad_lines="skip")
print(f"Loaded {len(df_orig)} rows")



Reading CSV file: /mnt/nvme1/proj/masa/traces/alibaba/cluster-trace-microservices-v2022/data/CallGraph/CallGraph_0.csv
Loaded 13331267 rows


In [ ]:
# Optional: Sample a fraction of traces to speed up analysis
# Set sample_fraction to 1.0 to use all traces, or a smaller value (e.g., 0.1 for 10%)
sample_fraction = 0.5
if sample_fraction < 1.0:
    df = sample_traces(df_orig, fraction=sample_fraction, random_state=42)
    print()

df.head()

Sampled 214,120 traces (10.0%) from 2,141,207 unique traces
Result: 1,342,531 rows from 13,331,267 original rows



,timestamp,traceid,service,rpc_id,rpctype,um,uminstanceid,interface,dm,dminstanceid,rt
18,168305,T_22121575692,S_85905920,0.1,rpc,MS_23205,MS_23205_POD_1206,1oNt-EK5Lm,MS_24094,MS_24094_POD_4972,3.0
42,56979,T_23776503347,S_73126470,0.1.2.5.33,mc,MS_7226,MS_7226_POD_136,rwvDNgNNj-,MS_14304,MS_14304_POD_139,1.0
45,57271,T_23776503347,S_73126470,0.1.2.9.5.6.1.3370129173,mq,MS_41914,MS_41914_POD_158,b4vXc2doXX,MS_54016,MS_54016_POD_26,1.0
48,56928,T_23776503347,S_73126470,0.1.2.4,mc,MS_39993,MS_39993_POD_83,hHGwtRtkGz,MS_37691,MS_37691_POD_248,1.0
58,41628,T_9925423159,S_18211983,0.1.1.1,mc,MS_29076,MS_29076_POD_110,OctX6xtC2C,MS_39556,MS_39556_POD_148,0.0


In [30]:
# Filter data and run parallel RPC detection
# First, filter out unknowns if needed
df_filtered = df[
    (df["um"].isin(["UNKNOWN", "UNAVAILABLE"]) == False) &
    (df["dm"].isin(["UNKNOWN", "UNAVAILABLE"]) == False)
].copy()

# Select only RPC rows (exclude UNKNOWN and mq types)
rpc_df = df_filtered[~df_filtered["rpctype"].isin(["UNKNOWN", "mq"])].copy()

print(f"Filtered to {len(rpc_df)} RPC rows from {len(df)} total rows")
print("\n" + "="*60)


Filtered to 1046840 RPC rows from 1342531 total rows



In [16]:
# Analyze sibling window overlap statistics
print_sibling_window_overlap_stats(rpc_df)

Sibling window overlap:
  Parent nodes with >=2 children: 158,665
  Sequential (non-overlapping) parents: 75,793 (47.77%)
  Overlapping parents: 82,872 (52.23%)
  Child calls under analyzed parents: 841,916
  Children in sequential groups: 219,480 (26.07%)
  Children in overlapping groups: 622,436 (73.93%)


In [25]:
# Analyze parent-child call patterns per service
call_patterns = analyze_parent_child_call_patterns(rpc_df)
print_call_pattern_summary(call_patterns)

Call Pattern Summary (aggregated per service):
  Total services analyzed: 3,271
  Total unique parent RPCs: 11,080
  Parents with only sequential calls: 7,521
  Parents with at least one fanout: 3,559
  Total unique pattern variations: 15,956
  Total call groups in most common patterns: 33,791
  Sequential call groups: 27,502
  Fanout groups: 6,289
  Total RPCs in fanouts: 16,997
  Fanout size statistics:
    Min: 2
    Max: 86
    Mean: 2.70
    Median: 2.00


In [26]:
# Show example call patterns for a specific service
if call_patterns:
    # Get first service as example
    example_service = list(call_patterns.keys())[0]
    example_patterns = call_patterns[example_service]
    
    print(f"\nExample call patterns for service: {example_service}")
    print(f"Number of unique parent RPCs: {len(example_patterns)}\n")
    
    # Show first few examples
    for i, (parent_rpc_id, parent_data) in enumerate(list(example_patterns.items())[:3]):
        print(f"Example {i+1}: Parent RPC {parent_rpc_id}")
        print(f"  Total occurrences across all traces: {parent_data['total_occurrences']}")
        print(f"  Unique pattern variations: {len(parent_data['patterns'])}")
        print(f"  Most common pattern (length {len(parent_data['most_common'])}):")
        
        pattern = parent_data['most_common']
        for j, item in enumerate(pattern):
            if isinstance(item, list):
                print(f"    [{j}] Fanout ({len(item)} parallel calls):")
                for k, rpc in enumerate(item):
                    print(f"      - RPC {rpc['rpc_id']} -> {rpc['dm']} ({rpc.get('interface', 'N/A')}) "
                          f"[{rpc['timestamp']:.1f} - {rpc['end_timestamp']:.1f}]")
            else:
                print(f"    [{j}] Sequential: RPC {item['rpc_id']} -> {item['dm']} "
                      f"({item.get('interface', 'N/A')}) [{item['timestamp']:.1f} - {item['end_timestamp']:.1f}]")
        
        # Show pattern frequency if there are multiple patterns
        if len(parent_data['patterns']) > 1:
            print(f"  Pattern frequencies:")
            for pattern_str, freq in sorted(parent_data['pattern_freq'].items(), key=lambda x: x[1], reverse=True)[:3]:
                print(f"    {pattern_str[:80]}... : {freq} occurrences")
        print()


Example call patterns for service: S_100192782
Number of unique parent RPCs: 3

Example 1: Parent RPC 0.1.1
  Total occurrences across all traces: 1
  Unique pattern variations: 1
  Most common pattern (length 1):
    [0] Sequential: RPC 0.1.1.1 -> MS_21381 (LmrQBbMncp) [84691.0 - 84720.0]

Example 2: Parent RPC 0.1.1.1
  Total occurrences across all traces: 1
  Unique pattern variations: 1
  Most common pattern (length 5):
    [0] Sequential: RPC 0.1.1.1.1 -> MS_24030 (I1UhotFy1C) [84700.0 - 84703.0]
    [1] Sequential: RPC 0.1.1.1.2 -> MS_24030 (BchceaiKa0) [84703.0 - 84707.0]
    [2] Fanout (2 parallel calls):
      - RPC 0.1.1.1.3 -> MS_39468 (dU3L0euv0H) [84711.0 - 84713.0]
      - RPC 0.1.1.1.3 -> MS_39468 (dU3L0euv0H) [84711.0 - 84714.0]
    [3] Fanout (2 parallel calls):
      - RPC 0.1.1.1.4 -> MS_26793 (IsaJNip1rZ) [84714.0 - 84717.0]
      - RPC 0.1.1.1.4 -> MS_26793 (IsaJNip1rZ) [84714.0 - 84716.0]
    [4] Fanout (2 parallel calls):
      - RPC 0.1.1.1.5 -> MS_26793 (c4Krr

In [27]:
# Get simplified call-pattern map (one pattern per parent, aggregated across traces)
simplified_patterns = get_simplified_call_patterns(call_patterns)

# Show summary of simplified patterns
print("Simplified Call-Pattern Maps (one per service):")
print(f"Number of services: {len(simplified_patterns)}\n")

for service, patterns in list(simplified_patterns.items())[:3]:
    print(f"Service: {service}")
    print(f"  Number of parent RPCs: {len(patterns)}")
    print(f"  Example parents: {list(patterns.keys())[:5]}")
    print()

Simplified Call-Pattern Maps (one per service):
Number of services: 3271

Service: S_100192782
  Number of parent RPCs: 3
  Example parents: ['0.1.1', '0.1.1.1', '0.1.1.1.3']

Service: S_100293104
  Number of parent RPCs: 3
  Example parents: ['0', '0.3', '0.4']

Service: S_100293121
  Number of parent RPCs: 1
  Example parents: ['0']

